In [5]:
import os
import gzip
import re
import json
import pandas as pd
from collections import defaultdict

# Define folder path
data_path = "../data"

# Define languages used in analysis
languages = ['en', 'de', 'hi', 'ja',]


In [8]:
import gzip
import os
import re
from collections import defaultdict

# Define the path to your data folder
data_path = "../data"

# Step 1: Extract page_id to title mapping
def extract_page_id_title_map(lang_code):
    filepath = os.path.join(data_path, f"{lang_code}wiki-latest-page.sql.gz")
    id_title_map = {}

    with gzip.open(filepath, 'rt', encoding='utf-8', errors='ignore') as file:
        for line in file:
            if line.startswith('INSERT INTO'):
                entries = re.findall(r"\((\d+),\d+,'([^']+)'", line)
                for page_id, title in entries:
                    id_title_map[int(page_id)] = title.replace(" ", "_")

    print(f"{lang_code}: Extracted {len(id_title_map)} page ID-title pairs.")
    return id_title_map

# Step 2: Extract pagelinks using the page ID map
def extract_pagelinks_with_titles(lang_code, id_title_map):
    filepath = os.path.join(data_path, f"{lang_code}wiki-latest-pagelinks.sql.gz")
    link_dict = defaultdict(list)
    link_count = 0

    with gzip.open(filepath, 'rt', encoding='utf-8', errors='ignore') as file:
        for line in file:
            if line.startswith('INSERT INTO'):
                entries = re.findall(r"\((\d+),0,(\d+)\)", line)
                for from_id, to_id in entries:
                    from_id, to_id = int(from_id), int(to_id)
                    if from_id in id_title_map and to_id in id_title_map:
                        from_title = id_title_map[from_id]
                        to_title = id_title_map[to_id]
                        link_dict[from_title].append(to_title)
                        link_count += 1

    print(f" {lang_code}: Extracted {link_count} links (using page titles).")
    return link_dict


In [9]:
lang_code_test = "en"
page_map = extract_page_id_title_map(lang_code_test)
link_dict = extract_pagelinks_with_titles(lang_code_test, page_map)

# Preview sample output
sample_output = {k: link_dict[k][:5] for k in list(link_dict.keys())[:5]}
sample_output



en: Extracted 63365894 page ID-title pairs.
 en: Extracted 542161274 links (using page titles).


{'Loyalty_program': ['AmoeboidTaxa',
  'Gheorghe_Hagi',
  'HighGerman',
  'Communications_in_Indonesia',
  'Ismailis'],
 'Economy_of_Mexico': ['AmoeboidTaxa',
  'Aquarius_(constellation)',
  'Apollo_12',
  'Adrastea_(moon)',
  'BinomialDistribution/Revisited'],
 'Walmart': ['AmoeboidTaxa',
  'Aquarius_(constellation)',
  'Akira_Kurosawa',
  'Apollo_1',
  'Arguments_for_the_existence_of_God'],
 'List_of_supermarket_chains': ['AmoeboidTaxa',
  'AfroAsiaticLanguages',
  'A',
  'Aquarius_(constellation)',
  'Ahab'],
 'Mexicana_de_Aviación_(1921–2010)': ['AmoeboidTaxa',
  'Carrier_battle_group',
  'Dune/Videogames',
  'Estimating_parameters',
  'Telecommunications_in_French_Guiana']}

In [13]:
import gzip
import os
import re
import pickle
from collections import defaultdict
from tqdm import tqdm  

# CONFIG

data_path = "../data"   # folder where .gz dumps are stored
output_path = "../data" # folder to save link graphs

# STEP 1: Extract page_id → title mapping
def extract_page_id_title_map(lang_code):
    filepath = os.path.join(data_path, f"{lang_code}wiki-latest-page.sql.gz")
    id_title_map = {}

    print(f" Reading: {filepath}")
    with gzip.open(filepath, 'rt', encoding='utf-8', errors='ignore') as file:
        for line in file:
            if line.startswith('INSERT INTO'):
                # Find tuples: (page_id, namespace, 'Title')
                entries = re.findall(r"\((\d+),\d+,'([^']+)'", line)
                for page_id, title in entries:
                    id_title_map[int(page_id)] = title.replace(" ", "_")
    print(f" {lang_code}: {len(id_title_map)} page ID-title mappings extracted.")
    return id_title_map

# STEP 2: Extract pagelinks using title map
def extract_pagelinks_with_titles(lang_code, id_title_map):
    filepath = os.path.join(data_path, f"{lang_code}wiki-latest-pagelinks.sql.gz")
    link_dict = defaultdict(list)
    link_count = 0

    print(f" Reading: {filepath}")
    with gzip.open(filepath, 'rt', encoding='utf-8', errors='ignore') as file:
        for line in tqdm(file, desc=f"Processing {lang_code} links"):
            if line.startswith('INSERT INTO'):
                # Find tuples: (from_id, namespace, to_id)
                entries = re.findall(r"\((\d+),0,(\d+)\)", line)
                for from_id, to_id in entries:
                    from_id, to_id = int(from_id), int(to_id)
                    if from_id in id_title_map and to_id in id_title_map:
                        from_title = id_title_map[from_id]
                        to_title = id_title_map[to_id]
                        link_dict[from_title].append(to_title)
                        link_count += 1

    print(f" {lang_code}: {link_count} total links extracted.")
    return link_dict

# STEP 3: Process one language
def process_and_save_language(lang_code):
    print(f"\n Starting: {lang_code}")
    id_map = extract_page_id_title_map(lang_code)
    link_dict = extract_pagelinks_with_titles(lang_code, id_map)

    output_file = os.path.join(output_path, f"{lang_code}_link_graph.pkl")
    with open(output_file, "wb") as f:
        pickle.dump(link_dict, f)

    print(f" Saved {lang_code} link graph to {output_file}")
    del id_map, link_dict

# RUN for all required languages
languages = ["en", "de", "hi", "ja"]  # new set
for lang in languages:
    process_and_save_language(lang)

print(" All languages processed successfully.")



 Starting: en
 Reading: ../data/enwiki-latest-page.sql.gz
 en: 63365894 page ID-title mappings extracted.
 Reading: ../data/enwiki-latest-pagelinks.sql.gz


Processing en links: 34321it [1:45:41,  5.41it/s] 


 en: 542161274 total links extracted.
 Saved en link graph to ../data/en_link_graph.pkl

 Starting: de
 Reading: ../data/dewiki-latest-page.sql.gz
 de: 8321984 page ID-title mappings extracted.
 Reading: ../data/dewiki-latest-pagelinks.sql.gz


Processing de links: 4768it [01:42, 46.32it/s] 


 de: 90805853 total links extracted.
 Saved de link graph to ../data/de_link_graph.pkl

 Starting: hi
 Reading: ../data/hiwiki-latest-page.sql.gz
 hi: 1373785 page ID-title mappings extracted.
 Reading: ../data/hiwiki-latest-pagelinks.sql.gz


Processing hi links: 732it [00:05, 134.52it/s]


 hi: 7894131 total links extracted.
 Saved hi link graph to ../data/hi_link_graph.pkl

 Starting: ja
 Reading: ../data/jawiki-latest-page.sql.gz
 ja: 4304144 page ID-title mappings extracted.
 Reading: ../data/jawiki-latest-pagelinks.sql.gz


Processing ja links: 3760it [01:35, 39.42it/s] 


 ja: 121971439 total links extracted.
 Saved ja link graph to ../data/ja_link_graph.pkl
 All languages processed successfully.


In [3]:
import pickle

with open("../data/en_link_graph.pkl", "rb") as f:
    en_links = pickle.load(f)

# Check number of nodes
print(f"🔗 English articles with outgoing links: {len(en_links)}")

# Preview a few entries
for k, v in list(en_links.items())[:5]:
    print(f"{k} → {v[:5]}")



🔗 English articles with outgoing links: 15834866
Loyalty_program → ['AmoeboidTaxa', 'Gheorghe_Hagi', 'HighGerman', 'Communications_in_Indonesia', 'Ismailis']
Economy_of_Mexico → ['AmoeboidTaxa', 'Aquarius_(constellation)', 'Apollo_12', 'Adrastea_(moon)', 'BinomialDistribution/Revisited']
Walmart → ['AmoeboidTaxa', 'Aquarius_(constellation)', 'Akira_Kurosawa', 'Apollo_1', 'Arguments_for_the_existence_of_God']
List_of_supermarket_chains → ['AmoeboidTaxa', 'AfroAsiaticLanguages', 'A', 'Aquarius_(constellation)', 'Ahab']
Mexicana_de_Aviación_(1921–2010) → ['AmoeboidTaxa', 'Carrier_battle_group', 'Dune/Videogames', 'Estimating_parameters', 'Telecommunications_in_French_Guiana']


In [14]:
import pandas as pd
import pickle

def extract_title_from_url(url):
    if pd.isna(url):
        return None
    return url.split("/")[-1].strip().replace(" ", "_")

# Load CSV
alignment_df = pd.read_csv("../data/qid.csv")

# Extract just the article title from the URL
alignment_df["en_title_clean"] = alignment_df["en_article"].apply(extract_title_from_url)

# Drop NaNs and make a set
en_titles = set(alignment_df["en_title_clean"].dropna())

# Load the link graph
with open("../data/en_link_graph.pkl", "rb") as f:
    en_links = pickle.load(f)

# Filter based on cleaned titles
filtered_en_links = {
    k: [t for t in v if t in en_titles]
    for k, v in en_links.items()
    if k in en_titles
}

print(f" Filtered English graph: {len(filtered_en_links)} nodes (after URL cleanup)")



 Filtered English graph: 1321 nodes (after URL cleanup)


In [15]:
with open("../data/en_link_graph.pkl","rb") as f:
    g = pickle.load(f)
print(list(g.keys())[:5])


['Loyalty_program', 'Economy_of_Mexico', 'Walmart', 'List_of_supermarket_chains', 'Mexicana_de_Aviación_(1921–2010)']


In [1]:
import os
import pickle
import pandas as pd
import urllib.parse

# Set your paths
data_path = "../data"
languages = ["en", "de", "hi", "ja"]

# Load alignment CSV
alignment_df = pd.read_csv(os.path.join(data_path, "qid.csv"))

# --- Step 1: Clean article titles from URLs ---
def extract_title_from_url(url):
    if pd.isna(url):
        return None
    raw_title = url.strip().split("/")[-1]
    return urllib.parse.unquote(raw_title).replace(" ", "_")

for lang in languages:
    col_name = f"{lang}_article"
    if col_name in alignment_df.columns:
        alignment_df[f"{lang}_title_clean"] = alignment_df[col_name].apply(extract_title_from_url)

# --- Step 2: Filter outbound edges for each language ---
for lang in languages:
    print(f"\n Processing: {lang}")

    graph_path = os.path.join(data_path, f"{lang}_link_graph.pkl")
    if not os.path.exists(graph_path):
        print(f" Skipped: {graph_path} not found")
        continue

    with open(graph_path, "rb") as f:
        link_graph = pickle.load(f)

    col_name = f"{lang}_title_clean"
    topic_titles = set(alignment_df[col_name].dropna())

    # Filter: keep only outbound edges from topic titles
    filtered_graph = {
        node: links
        for node, links in link_graph.items()
        if node in topic_titles
    }

    # Ensure isolated topic nodes are included with no edges
    for title in topic_titles:
        filtered_graph.setdefault(title, [])

    print(f" {lang}: {len(filtered_graph)} nodes, "
          f"{sum(len(v) for v in filtered_graph.values())} edges")

    # Save filtered graph
    output_path = os.path.join(data_path, f"{lang}_filtered_graph.pkl")
    with open(output_path, "wb") as f_out:
        pickle.dump(filtered_graph, f_out)

print("\n Filtering complete. Use the filtered graphs for bias analysis.")



🔁 Processing: en
🎯 en: 1360 nodes, 151396 edges

🔁 Processing: de
🎯 de: 929 nodes, 35977 edges

🔁 Processing: hi
🎯 hi: 530 nodes, 23160 edges

🔁 Processing: ja
🎯 ja: 751 nodes, 74097 edges

✅ Filtering complete. Use the filtered graphs for bias analysis.


In [12]:
import os, pickle, networkx as nx, pandas as pd, urllib.parse

DATA_PATH = "../data"
OUT_CSV = os.path.join(DATA_PATH, "network_metrics.csv")
LANGS = ["en","de","hi","ja"]

def norm_title(s):
    if s is None: return None
    return urllib.parse.unquote(str(s)).strip().replace(" ", "_")

all_metrics = []

for lang in LANGS:
    # Prefer expanded
    pref = os.path.join(DATA_PATH, f"{lang}_filtered_graph_expanded.pkl")
    alt  = os.path.join(DATA_PATH, f"{lang}_filtered_graph.pkl")
    fp = pref if os.path.exists(pref) else alt
    if not os.path.exists(fp):
        print(f" {lang}: neither expanded nor filtered graph found.")
        continue

    with open(fp, "rb") as f:
        link_dict = pickle.load(f)

    G = nx.DiGraph()
    G.add_nodes_from(link_dict.keys())
    for src, tgts in link_dict.items():
        for tgt in tgts:
            if tgt not in G: G.add_node(tgt)
            G.add_edge(src, tgt)

    try:
        pr = nx.pagerank_scipy(G) if G.number_of_edges()>0 else {n:0.0 for n in G.nodes()}
    except Exception:
        pr = nx.pagerank(G) if G.number_of_edges()>0 else {n:0.0 for n in G.nodes()}

    indeg = dict(G.in_degree())
    outdeg = dict(G.out_degree())

    if G.number_of_nodes() > 2000:
        btw = nx.betweenness_centrality(G, k=800, seed=42)
    else:
        btw = nx.betweenness_centrality(G)

    try:
        eig = nx.eigenvector_centrality(G, max_iter=300) if G.number_of_edges()>0 else {n:0.0 for n in G.nodes()}
    except Exception:
        eig = {n:0.0 for n in G.nodes()}

    clust = nx.clustering(G.to_undirected()) if G.number_of_edges()>0 else {n:0.0 for n in G.nodes()}

    df = pd.DataFrame({
        "language": lang,
        "title": list(G.nodes()),
        "title_clean": [norm_title(n) for n in G.nodes()],
        "pagerank": [pr.get(n,0.0) for n in G.nodes()],
        "in_degree": [indeg.get(n,0) for n in G.nodes()],
        "out_degree": [outdeg.get(n,0) for n in G.nodes()],
        "betweenness": [btw.get(n,0.0) for n in G.nodes()],
        "eigenvector": [eig.get(n,0.0) for n in G.nodes()],
        "clustering": [clust.get(n,0.0) for n in G.nodes()],
    })
    all_metrics.append(df)
    print(f"{lang}: nodes={G.number_of_nodes()}, edges={G.number_of_edges()} (source: {'expanded' if fp==pref else 'filtered'})")

pd.concat(all_metrics, ignore_index=True).to_csv(OUT_CSV, index=False)
print(f" saved metrics → {OUT_CSV}")


en: nodes=408260, edges=588918 (source: expanded)
de: nodes=142154, edges=188768 (source: expanded)
hi: nodes=25090, edges=54315 (source: expanded)
ja: nodes=142814, edges=233947 (source: expanded)
 saved metrics → ../data/network_metrics.csv


In [3]:
import pandas as pd

# ------------------------------
# Load raw unfiltered dataset
# ------------------------------
query_file = "../data/query2.csv"  # adjust if needed
query_df = pd.read_csv(query_file)

print("Original columns:", query_df.columns.tolist())
print(query_df.head(5))

# ------------------------------
# STEP 1: Drop unnecessary columns
# ------------------------------
columns_to_drop = ["field_of_work", "occupation"]
query_df = query_df.drop(columns=[col for col in columns_to_drop if col in query_df.columns])

# ------------------------------
# STEP 2: Identify language columns
# ------------------------------
language_columns = [col for col in query_df.columns if col.endswith('_article')]

# ------------------------------
# STEP 3: Create langs_present for each QID
# ------------------------------
def get_langs_present(row):
    langs = []
    for col in language_columns:
        if pd.notna(row[col]) and str(row[col]).strip() != "":
            langs.append(col.split("_")[0])  # get 'en' from 'en_article'
    return ",".join(sorted(langs))

query_df["langs_present"] = query_df.apply(get_langs_present, axis=1)

# ------------------------------
# STEP 4: Melt into (qid, language, title)
# ------------------------------
query_melted = query_df.melt(
    id_vars=['qid', 'langs_present'],  # keep langs_present for every row
    value_vars=language_columns,
    var_name='language',
    value_name='title'
)

# Remove missing titles
query_melted = query_melted.dropna(subset=['title'])

# Clean language codes (e.g., en_article -> en)
query_melted['language'] = query_melted['language'].str.replace('_article', '', regex=False)

# ------------------------------
# STEP 5: Save the Cleaned File
# ------------------------------
converted_file = "../data/query_converted.csv"
query_melted.to_csv(converted_file, index=False)

print(f"✅ Saved converted query: {converted_file} with shape {query_melted.shape}")
print(query_melted.head(10))


Original columns: ['qid', 'label_en', 'en_article', 'de_article', 'es_article', 'fr_article', 'it_article', 'pt_article', 'field_of_work', 'occupation']
         qid                   label_en                 en_article  \
0  Q32586937  Category:Lists of islands  Category:Lists_of_islands   
1    Q622417    Saint-Pons-de-Thomières    Saint-Pons-de-Thomières   
2   Q6402462         Category:User fi-N         Category:User_fi-N   
3  Q25573373          biosphere reserve          Biosphere_reserve   
4   Q1035160                Hello Nasty                Hello_Nasty   

                        de_article  \
0         Kategorie:Liste_(Inseln)   
1          Saint-Pons-de-Thomières   
2  Kategorie:Benutzer:Sprache_fi-M   
3               Biosphärenreservat   
4                      Hello_Nasty   

                                          es_article  \
0                             Categoría:Anexos:Islas   
1                            Saint-Pons-de-Thomières   
2  Categoría:Wikipedia:Wikipe

In [7]:
import os
import pandas as pd

# Paths
inp = "../data/property.csv"
out_dir = "../data"
os.makedirs(out_dir, exist_ok=True)

# Load 
df = pd.read_csv(inp)

# Normalized copy for mapping (lower/strip)
def norm(s):
    return str(s).strip().lower() if pd.notna(s) else s

norm_df = df.copy()
for col in norm_df.columns:
    norm_df[col] = norm_df[col].apply(norm)

# 5-category OCCUPATION mapping 
def map_occupation_5(s: str):
    if not isinstance(s, str):
        return None
    f = s.lower().strip()

    # 1) Science (Science & Engineering + Technology & Media)
    if any(k in f for k in [
        # science/eng
        "physicist","chemist","biolog","astronom","engineer","researcher",
        "inventor","mathematic","ecolog","geolog","environmental",
        # tech/media
        "computer scientist","software","programmer","developer","data scientist",
        "journalist","reporter","editor","broadcaster","tv presenter",
        "television presenter","media","correspondent","news anchor"
    ]):
        return "Science"

    # 2) Economics & Politics
    if any(k in f for k in [
        # econ/business
        "economist","entrepreneur","business","investor","banker",
        "industrialist","merchant","finance","economics","executive","ceo",
        # politics/governance
        "politic","president","prime minister","senator","mayor","minister",
        "chancellor","diplomat","governor","mp","parliament","legislat","statesman"
    ]):
        return "Economics & Politics"

    # 3) Sports
    if any(k in f for k in [
        "football","association football player","cricket","cricketer","athlet",
        "basketball","runner","cyclist","sport cyclist","swimm","olymp",
        "tennis","hockey","badminton","wrestl","boxer","sprinter","goalkeeper","midfielder"
    ]):
        return "Sports"

    # 4) Arts, Literature & Humanities
    if any(k in f for k in [
        # arts/entertainment
        "actor","actress","film actor","television actor","stage actor","director",
        "film director","musician","singer","composer","dancer","painter","artist",
        "playwright","entertainer","producer","screenwriter","filmmaker",
        # literature & philosophy
        "writer","novelist","essayist","poet","philosopher","author","literature",
        # humanities & social sciences / academia
        "historian","sociolog","anthropolog","archaeolog","psycholog","geograph",
        "theolog","university teacher","professor","lecturer","academic","research fellow"
    ]):
        return "Arts, Literature & Humanities"

    return None

# 5-category INSTANCE-OF mapping 
def map_instance_5(s: str):
    if not isinstance(s, str):
        return None
    f = s.lower().strip()

    # 1) Person
    if any(k in f for k in ["human","person","people"]):
        return "Person"

    # 2) Place
    if any(k in f for k in [
        "country","state","nation","city","town","village","settlement",
        "municipality","region","province","county","district","geopolitical"
    ]):
        return "Place"

    # 3) Organisation
    if any(k in f for k in [
        "organization","organisation","company","institution","university",
        "museum","association","club","business","ngo","foundation","agency"
    ]):
        return "Organisation"

    # 4) Creative Work (incl. events for simplicity)
    if any(k in f for k in [
        # works
        "film","movie","television series","tv series","episode",
        "album","song","single","composition","painting","novel","book",
        "poem","literary work","creative work",
        # events folded here
        "event","war","battle","election","festival","tournament","competition"
    ]):
        return "Creative Work"

    return None

# Build per-property outputs 

# Occupation mapped → item, itemLabel, instanceOfLabel, occupation_category
if "occupationLabel" in df.columns:
    occ = df[["item","itemLabel","instanceOfLabel","occupationLabel"]].copy()
    occ_norm = norm_df[["occupationLabel"]].copy()
    occ["occupation_category"] = occ_norm["occupationLabel"].apply(map_occupation_5)
    # Drop rows without occupation OR unmapped
    occ = occ.dropna(subset=["occupationLabel","occupation_category"])
    occ = occ[["item","itemLabel","instanceOfLabel","occupation_category"]]
    occ_out = os.path.join(out_dir, "../data/occupation_mapped.csv")
    occ.to_csv(occ_out, index=False)
    print(f" Saved {occ.shape[0]} rows → {occ_out}")

# Instance mapped → item, itemLabel, instanceOfLabel, instance_category (5 buckets incl. Other)
if "instanceOfLabel" in df.columns:
    inst = df[["item","itemLabel","instanceOfLabel"]].copy()
    inst_norm = norm_df[["instanceOfLabel"]].copy()
    inst["instance_category"] = inst_norm["instanceOfLabel"].apply(map_instance_5)
    # Keep rows with instanceOfLabel present; 'Other' is allowed
    inst = inst.dropna(subset=["instanceOfLabel"])
    inst = inst[["item","itemLabel","instanceOfLabel","instance_category"]]
    inst_out = os.path.join(out_dir, "../data/instance_mapped.csv")
    inst.to_csv(inst_out, index=False)
    print(f" Saved {inst.shape[0]} rows → {inst_out}")

# Citizenship raw → item, itemLabel, instanceOfLabel, citizenshipLabel
if "citizenshipLabel" in df.columns:
    cit = df[["item","itemLabel","instanceOfLabel","citizenshipLabel"]].copy()
    cit = cit.dropna(subset=["citizenshipLabel"])
    cit_out = os.path.join(out_dir, "../data/citizenship.csv")
    cit.to_csv(cit_out, index=False)
    print(f"Saved {cit.shape[0]} rows → {cit_out}")

# Origin raw → item, itemLabel, instanceOfLabel, originLabel
if "originLabel" in df.columns:
    ori = df[["item","itemLabel","instanceOfLabel","originLabel"]].copy()
    ori = ori.dropna(subset=["originLabel"])
    ori_out = os.path.join(out_dir, "../data/origin.csv")
    ori.to_csv(ori_out, index=False)
    print(f" Saved {ori.shape[0]} rows → {ori_out}")

# Combined file (merge what exists)
combined = df[["item","itemLabel","instanceOfLabel"]].drop_duplicates().copy()

if 'occ' in locals() and not occ.empty:
    combined = combined.merge(occ, on=["item","itemLabel","instanceOfLabel"], how="left")

if 'inst' in locals() and not inst.empty:
    combined = combined.merge(inst[["item","itemLabel","instanceOfLabel","instance_category"]],
                              on=["item","itemLabel","instanceOfLabel"], how="left")

if 'cit' in locals() and not cit.empty:
    combined = combined.merge(cit, on=["item","itemLabel","instanceOfLabel"], how="left")

if 'ori' in locals() and not ori.empty:
    combined = combined.merge(ori, on=["item","itemLabel","instanceOfLabel"], how="left")

# Deduplicate any duplicate columns if introduced
combined = combined.loc[:, ~combined.columns.duplicated()]

combined_out = os.path.join(out_dir, "../data/properties_mapped_combined.csv")
combined.to_csv(combined_out, index=False)
print(f"Combined file → {combined_out}")


 Saved 607 rows → ../data/../data/occupation_mapped.csv
 Saved 2510 rows → ../data/../data/instance_mapped.csv
Saved 929 rows → ../data/../data/citizenship.csv
 Saved 146 rows → ../data/../data/origin.csv
Combined file → ../data/../data/properties_mapped_combined.csv
